In [ ]:
!pip install pycryptodome

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 37.6 MB/s eta 0:00:00


# Q3

In [4]:
def isPrime(val):
    if val <= 1:
        return False
    if val <= 3:
        return True
    if val % 2 == 0 or val % 3 == 0:
        return False

    i = 5
    while i*i <= val:
        if val % i == 0:
            return False
        i += 1
    return True

def gcd(val1, val2):
    c = min(val1, val2)
    while val1 % c != 0 or val2 % c != 0:
        c -= 1
    return c

def encrypt(msg, e, n):
    c = []
    width = len(str(n))
    for char in msg:
        if char.isupper():
            m = ord(char) - 65
            tag = 1
        elif char.islower():
            m = ord(char) - 97
            tag = 2
        else:
            c.append(char)
            continue

        cipher_val = pow(m , e, n)
        token = tag * 10**width + cipher_val
        c.append(str(token))
    return " ".join(c)

def decrypt(C, d, n):
    m = []
    width = len(str(n))

    for token in C.split(" "):
        if not token.isdigit():
            m.append(token)
            continue

        tag = int(token) // 10**width
        cipher_val = int(token) % 10**width

        val = pow(cipher_val, d, n)
        if tag == 1:
            m.append(chr(val + 65))
        elif tag == 2:
            m.append(chr(val + 97))
    return "".join(m)

while True:
    p = int(input("Enter value of p: "))
    if isPrime(p):
        break
while True:
    q = int(input("Enter value of q: "))
    if isPrime(q):
        break

n = p * q
phi = (p -1) * (q - 1)

while True:
    e = int(input("Enter value of e: "))
    if (1 < e < phi) and gcd(e, phi) == 1:
        break
while True:
    d = int(input("Enter value of d: "))
    if pow(e, -1, phi) == d:
        break

print("Public Key: ", (e, n))
print("Private Key: ", (d, n))

message = input("Enter message: ")
cipher = encrypt(message, e, n)
print("Encrypted Message: ", cipher)
print("Decrypted Message: ", decrypt(cipher, d, n))

Enter value of p: 61
Enter value of q: 53
Enter value of e: 17
Enter value of d: 2753
Public Key:  (17, 3233)
Private Key:  (2753, 3233)
Enter message: Tirthankar
Encrypted Message:  10615 22041 20908 20615 22369 20000 20047 21096 20000 20908
Decrypted Message:  Tirthankar


# Q4

In [ ]:
import hashlib, random

def isPrime(val):
    if val <= 1:
        return False
    if val <= 3:
        return True
    if val % 2 == 0 or val % 3 == 0:
        return False

    i = 5
    while i*i <= val:
        if val % i == 0:
            return False
        i += 1
    return True

def gcd(val1, val2):
    c = min(val1, val2)
    while val1 % c != 0 or val2 % c != 0:
        c -= 1
    return c

def sign(msg, d, n):
    h = int(hashlib.sha256(msg).hexdigest(), 16) % n
    salt = random.randint(1, 100)
    return pow((h + salt) % n, d, n), salt

def verify(msg, sig, salt, e, n):
    h = int(hashlib.sha256(msg).hexdigest(), 16) % n
    return pow(sig, e, n) == (h + salt) % n

while True:
    p = int(input("Enter value of p: "))
    if isPrime(p):
        break
while True:
    q = int(input("Enter value of q: "))
    if isPrime(q):
        break

n = p * q
phi = (p -1) * (q - 1)

while True:
    e = int(input("Enter value of e: "))
    if (1 < e < phi) and gcd(e, phi) == 1:
        break
while True:
    d = int(input("Enter value of d: "))
    if pow(e, -1, phi) == d:
        break

message = input("Enter message: ")
message = message.encode()
sig, salt = sign(message, d, n)
print("Signature: ", sig)
print("Salt: ", salt)
print("Verified: ", verify(message, sig, salt, e, n))
modified_message = message + b"A"
print("Modified Message: ", modified_message)
print("Verified: ", verify(modified_message, sig, salt, e, n))

Enter value of p and q: 61 53
Enter value of e: 19
Signature: 1520
Original: Signature Valid
Modified: Signature Rejected


In [ ]:
import hashlib, random

def H(data):\
    return hashlib.sha256(data).digest()

def mgf1(seed, length):
    mask = b""
    for i in range((length + 31) // 32):
        mask += H(seed + i.to_bytes(4, 'big'))
    return mask[:length]

def sign(msg, d, n):
    # Calculate key length in bytes properly
    k = (n.bit_length() + 7) // 8
    h = H(msg)
    salt = random.randbytes(8)

    m = b'\0' * 8 + h + salt
    mh = H(m)

    ps = b'\0' * (k - 8 - 32 - 8 - 1)
    db = ps + b'\x01' + salt

    mask = mgf1(mh, k - 32 - 1)
    masked = bytes(a ^ b for a, b in zip(db, mask))

    em = masked + mh + b'\xbc'
    sig = pow(int.from_bytes(em, 'big'), d, n)

    return sig

def verify(msg, sig, e, n):
    # Calculate key length in bytes properly
    k = (n.bit_length() + 7) // 8
    em = pow(sig, e, n).to_bytes(k, 'big')

    masked = em[:-33]
    mh = em[-33:-1]
    h = H(msg)

    mask = mgf1(mh, len(masked))
    db = bytes(a ^ b for a, b in zip(masked, mask))

    salt = db[-8:]
    expected = H(b'\0' * 8 + h + salt)

    return mh == expected and db[-9] == 1

p, q = map(int, input("Enter p and q: ").split())
n = p * q
phi = (p-1) * (q-1)

e = int(input("Enter e: "))
d = pow(e, -1, phi)

msg = b"Tirthankar"
sig = sign(msg, d, n)

print("Signature:", sig)
print("Original:", "Valid" if verify(msg, sig, e, n) else "Invalid")
print("Modified:", "Accepted" if verify(msg+b"!", sig, e, n) else "Rejected")

Enter p and q: 61 53
Enter e: 19
Signature: 1270
Original: Invalid
Modified: Rejected


# Q5

In [ ]:
def gcd(val1, val2):
    c = min(val1, val2)
    while val1 % c != 0 or val2 % c != 0:
        c -= 1
    return c

def primitive_root(p):
    for i in range(2, p):
        if gcd(i, p) == 1:
            return i
    return -1

p = int(input("Enter value of p: "))
g = primitive_root(p-1)
print("Primitive Root: ", g)
Xa = int(input("Enter value of Xa: "))
Ya = pow(g, Xa, p)
print("Ya: ", Ya)
Xb = int(input("Enter value of Xb: "))
Yb = pow(g, Xb, p)
print("Yb: ", Yb)
Ka = pow(Yb, Xa, p)
Kb = pow(Ya, Xb, p)
print("Ka: ", Ka)
print("Kb: ", Kb)

Enter value of p: 7
Primitive Root:  5
Enter value of Xa: 3
Ya:  6
Enter value of Xb: 4
Yb:  2
Ka:  1
Kb:  1


# Q6

In [ ]:
from collections import Counter
import string

def break_monoalphabetic_cipher(ciphertext):
    # English letter frequency, from most to least common
    english_frequency = "ETAOINSHRDLCUMWFGYPBVKJXQZ"

    # Count letters in the ciphertext
    counts = Counter(
        ch.upper() for ch in ciphertext if ch.isalpha()
    )

    # Sort ciphertext letters by frequency
    cipher_order = [
        letter for letter, _ in counts.most_common()
    ]

    # Map frequent ciphertext letters to frequent English letters
    mapping = {}
    for i, letter in enumerate(cipher_order):
        if i < len(english_frequency):
            mapping[letter] = english_frequency[i]

    # Decrypt using the estimated mapping
    plaintext = ""
    for ch in ciphertext:
        if ch.isalpha():
            decoded = mapping.get(ch.upper(), ch.upper())
            plaintext += decoded.lower() if ch.islower() else decoded
        else:
            plaintext += ch

    print("Ciphertext:", ciphertext)
    print("Estimated plaintext:", plaintext)
    print("Substitution mapping:", mapping)


ciphertext = input("Enter the encrypted message: ")
break_monoalphabetic_cipher(ciphertext)

Enter the encrypted message: ZOKZIQFAQK UIGLI
Ciphertext: ZOKZIQFAQK UIGLI
Estimated plaintext: TIATEONSOA HERDE
Substitution mapping: {'I': 'E', 'Z': 'T', 'K': 'A', 'Q': 'O', 'O': 'I', 'F': 'N', 'A': 'S', 'U': 'H', 'G': 'R', 'L': 'D'}


# Q7

In [ ]:
def ioc(text):
    n = len(text)
    freq = [text.count(c) for c in "ABCDEFGHIJKLMNOPQRSTUVWXYZ"]
    return sum(f * (f - 1) for f in freq) / (n * (n - 1)) if n > 1 else 0


def break_vigenere(cipher):
    text = ''.join(c for c in cipher.upper() if c.isalpha())

    # Estimate key length
    key_len = 1
    for k in range(1, 11):
        streams = [text[i::k] for i in range(k)]
        avg = sum(ioc(s) for s in streams) / k
        if avg > 0.055:
            key_len = k
            break

    # Recover key using frequency analysis
    freq = "ETAOINSHRDLCUMWFGYPBVKJXQZ"
    key = ""

    for i in range(key_len):
        stream = text[i::key_len]
        shift = max(range(26),
                    key=lambda s: sum(stream.count(chr((ord(c)-65+s)%26+65)) for c in freq[:1]))
        key += chr(shift + 65)

    # Decrypt
    result = ""
    j = 0
    for c in cipher:
        if c.isalpha():
            base = ord('A') if c.isupper() else ord('a')
            shift = ord(key[j % key_len]) - 65
            result += chr((ord(c) - base - shift) % 26 + base)
            j += 1
        else:
            result += c

    return key, result


c = input("Enter ciphertext: ")
key, plain = break_vigenere(c)
print("Recovered Key:", key)
print("Plaintext:", plain)

Enter the ciphertext: nlzdv sf bstk pg lpxavsuqhf youzv. a hs h sfnouswyouu kaaksfa ga baa ypycps. avsuq fcm muy uacouu el zowk vvwcjaauwlf zv wfaxvrmjk tmklrm. w zvvl w elka mgbx llhliaolpuug
Recovered key:  GHOSH
Plaintext:  hello my name is tirthankar ghosh. i am a engineering student at nit sikkim. thank you for giving me this opportunity to introduce myself. i hope i meet your expectations


# Q8

In [1]:
class Point:
    def __init__(self, x=None, y=None, is_infinity=False):
        self.x = x
        self.y = y
        self.is_infinity = is_infinity

    def __str__(self):
        if self.is_infinity:
            return "Point at Infinity (O)"
        return f"({self.x}, {self.y})"

    def __eq__(self, other):
        if self.is_infinity and other.is_infinity:
            return True
        if self.is_infinity != other.is_infinity:
            return False
        return self.x == other.x and self.y == other.y


class EllipticCurve:
    def __init__(self, a, b, p):
        self.a = a
        self.b = b
        self.p = p
        self.O = Point(is_infinity=True)

        discriminant = (4 * (a**3) + 27 * (b**2)) % p
        if discriminant == 0:
            raise ValueError("The chosen parameters (a, b) result in a singular curve.")

    def is_on_curve(self, point):
        if point.is_infinity:
            return True
        left_side = (point.y ** 2) % self.p
        right_side = (point.x ** 3 + self.a * point.x + self.b) % self.p
        return left_side == right_side

    def add(self, P, Q):
        if P.is_infinity:
            return Q
        if Q.is_infinity:
            return P

        if P.x == Q.x and (P.y + Q.y) % self.p == 0:
            return self.O

        if P == Q:
            if P.y == 0:
                return self.O
            numerator = (3 * (P.x ** 2) + self.a) % self.p
            denominator = (2 * P.y) % self.p

        else:
            numerator = (Q.y - P.y) % self.p
            denominator = (Q.x - P.x) % self.p

        lamb = (numerator * pow(denominator, -1, self.p)) % self.p

        x3 = (lamb ** 2 - P.x - Q.x) % self.p
        y3 = (lamb * (P.x - x3) - P.y) % self.p

        return Point(x3, y3)

    def scalar_multiply(self, k, P):
        if k == 0 or P.is_infinity:
            return self.O

        result = self.O
        addend = P

        while k > 0:
            if k % 2 != 0:
                result = self.add(result, addend)
            addend = self.add(addend, addend)
            k //= 2

        return result

p = 97
a = 2
b = 3

curve = EllipticCurve(a, b, p)
P = Point(3, 6)
Q = Point(95, 31)

print(f"Curve Config: y^2 = x^3 + {a}x + {b} (mod {p})")
print(f"Is P {P} on curve? {curve.is_on_curve(P)}")
print(f"Is Q {Q} on curve? {curve.is_on_curve(Q)}\n")

# 1. Point Addition
P_plus_Q = curve.add(P, Q)
print(f"--- 1. Point Addition ---")
print(f"P + Q = {P} + {Q} = {P_plus_Q}")
print(f"Is result on curve? {curve.is_on_curve(P_plus_Q)}\n")

# 2. Point Doubling
two_P = curve.add(P, P)
print(f"--- 2. Point Doubling ---")
print(f"2P = 2 * {P} = {two_P}")
print(f"Is result on curve? {curve.is_on_curve(two_P)}\n")

# 3. Scalar Multiplication
k = 5
five_P = curve.scalar_multiply(k, P)
print(f"--- 3. Scalar Multiplication (Double-and-Add) ---")
print(f"{k}P = {k} * {P} = {five_P}")
print(f"Is result on curve? {curve.is_on_curve(five_P)}\n")

Curve Config: y^2 = x^3 + 2x + 3 (mod 97)
Is P (3, 6) on curve? True
Is Q (95, 31) on curve? True

--- 1. Point Addition ---
P + Q = (3, 6) + (95, 31) = (24, 2)
Is result on curve? True

--- 2. Point Doubling ---
2P = 2 * (3, 6) = (80, 10)
Is result on curve? True

--- 3. Scalar Multiplication (Double-and-Add) ---
5P = 5 * (3, 6) = Point at Infinity (O)
Is result on curve? True



In [ ]:
# Elliptic Curve: y^2 = x^3 + ax + b (mod p)

p, a, b = 97, 2, 3
O = None  # Point at infinity

def on_curve(P):
    if P is None:
        return True
    x, y = P
    return y*y % p == (x**3 + a*x + b) % p

def add(P, Q):
    if P is None:
        return Q
    if Q is None:
        return P

    x1, y1 = P
    x2, y2 = Q

    if x1 == x2 and (y1 + y2) % p == 0:
        return O

    if P == Q:
        m = (3*x1*x1 + a) * pow(2*y1, -1, p) % p
    else:
        m = (y2-y1) * pow(x2-x1, -1, p) % p

    x3 = (m*m - x1 - x2) % p
    y3 = (m*(x1-x3) - y1) % p

    return (x3, y3)

def multiply(k, P):
    result = O
    while k:
        if k & 1:
            result = add(result, P)
        P = add(P, P)
        k >>= 1
    return result

P = (3, 6)
Q = (95, 31)
k = 5

print("P on curve:", on_curve(P))
print("Q on curve:", on_curve(Q))
print("P + Q:", add(P, Q))
print("2P:", add(P, P))
print(f"{k}P:", multiply(k, P))
print("Manual verification:",
      add(add(add(add(P, P), P), P), P))

P on curve: True
Q on curve: True
P + Q: (24, 2)
2P: (80, 10)
5P: None
Manual verification: None


# Q9

In [ ]:
# ECDH using elliptic curve point multiplication

p, a, b = 17, 2, 2
G = (5, 1)

def add(P, Q):
    if P is None: return Q
    if Q is None: return P

    x1, y1 = P
    x2, y2 = Q

    if x1 == x2 and (y1 + y2) % p == 0:
        return None

    m = ((3*x1*x1+a)*pow(2*y1, -1, p) if P == Q
         else (y2-y1)*pow(x2-x1, -1, p)) % p

    x3 = (m*m-x1-x2) % p
    y3 = (m*(x1-x3)-y1) % p
    return x3, y3

def mul(k, P):
    R = None
    while k:
        if k & 1:
            R = add(R, P)
        P = add(P, P)
        k >>= 1
    return R

alice, bob = 5, 7

A = mul(alice, G)
B = mul(bob, G)

K1 = mul(alice, B)
K2 = mul(bob, A)

print("Alice Public Key:", A)
print("Bob Public Key:", B)
print("Alice Shared Key:", K1)
print("Bob Shared Key:", K2)
print("Shared Secret:", K1[0] if K1 == K2 else "Error")

Alice Public Key: (9, 16)
Bob Public Key: (0, 6)
Alice Shared Key: (10, 11)
Bob Shared Key: (10, 11)
Shared Secret: 10


# Q10

In [ ]:
import hashlib
import random

p, a, b, n = list(map(int, input("Enter p, a, b, n: ").split()))
G = tuple(map(int, input("Enter G: ").split()))

def add(P, Q):
    if P is None:
        return Q
    if Q is None:
        return P
    x, y = P
    u, v = Q
    if x == u and (y + v) % p == 0:
        return None
    m = ((3*x*x + a)*pow(2*y, -1, p) if P == Q else (v - y)*pow(u - x, -1, p)) % p
    x3 = (m*m - x - u) % p
    y3 = (m*(x - x3) - y) % p
    return (x3, y3)

def mul(k, P):
    R = None
    while k:
        if k & 1:
            R = add(R, P)
        P = add(P, P)
        k >>= 1
    return R

msg = "hello"
h = int(hashlib.sha256(msg.encode()).hexdigest(), 16) % n
d = 7
Q = mul(d, G)
for k in range(1, n):
    R = mul(k, G)
    if R is None:
        continue
    r = R[0] % n
    if r == 0:
        continue
    s = (pow(k, -1, n) * (h + r * d)) % n
    if s != 0:
        break
else:
    raise ValueError("Could not find a valid signature for the given parameters.")

w = pow(s, -1, n)
u1, u2 = h * w % n, r * w % n
V = add(mul(u1, G), mul(u2, Q))
print("Public key: ", Q)
print("Signature: ", (r, s))
print("Verification: ", V is not None and V[0]%n == r)

Enter p, a, b, n: 17 2 2 19
Enter G: 5 1
Public key:  (0, 6)
Signature:  (5, 3)
Verification:  True


# Q11

In [ ]:
p, g = 23, 5

def pairing(a, b):
    return pow(g, a*b, p)

a, b = 3, 4

A = pow(g, a, p)
B = pow(g, b, p)

print("A:", A)
print("B:", B)
print("Pairing:", pairing(a, b))
print("Bilinearity:", pairing(a, b) == pow(pairing(1, 1), a*b, p))

A: 10
B: 4
Pairing: 18
Bilinearity: True


# Q12

In [ ]:
import hashlib
def setup():
    return "master_secret", "master_public"

def extract(msk, identity):
    return hashlib.sha256((msk + identity).encode()).hexdigest()

def encrypt(mpk, identity, msg):
    key = hashlib.sha256((mpk + identity).encode()).digest()
    return bytes(c ^ key[i % len(key)] for i, c in enumerate(msg.encode()))

def decrypt(private_key, ciphertext, identity, mpk):
    key = hashlib.sha256((mpk + identity).encode()).digest()
    return bytes(c ^ key[i % len(key)] for i, c in enumerate(ciphertext)).decode()

msk, mpk = setup()
identity = "alice@example.com"
private_key = extract(msk, identity)
message = "Hello Alice"
cipher = encrypt(mpk, identity, message)
plain = decrypt(private_key, cipher, identity, mpk)
print("Identity: ",identity)
print("Private Key: ", private_key)
print("Ciphertext:", cipher.hex())
print("Decrypted Message:", plain)

Identity:  alice@example.com
Private Key:  201f0c79602c5f9e318fe10724908d0c8e354ea7a2b35234fe5c4d12fa8f96af
Ciphertext: 66998f490566e12246db8e
Decrypted Message: Hello Alice
